# Q2 - FFA "Kinobesucher*innen 2025": Exploration

**Source:** Filmförderungsanstalt (FFA), *Kinobesucher\*innen 2025* (PDF), used with permission of the FFA and with attribution.

This notebook explores the four foles of source Q2. The raw data in `data/raw/ffa` is only read, never modified. 

### 1: Check that Python and panadas work

In [39]:
#Show the Python and pandas versions
# If two version number appear, the setup works 
import sys 
print(sys.version)

import pandas as pd
print(pd.__version__)

3.13.1 (v3.13.1:06714517797, Dec  3 2024, 14:00:22) [Clang 15.0.0 (clang-1500.3.9.4)]
3.1.1


### 2: Find out where the Noteboook is running 

In [40]:
# Show the folder the notebook is running in 
# The path to the data depends on this folder
import os 
print(os.getcwd())

/Users/aqsashaikh/data-integration-projekt/notebooks/q2_ffa_visitors


### 3: Check that the data folder can be found  

In [41]:

# List all files in the data folder
# "../../" means: go up two folders (aus notebooks/q2_ffa_visitors/ back to the project root)
os.listdir("../../data/raw/ffa")

['ffa_tickets_total_arthouse_2001_2025.csv',
 'ffa_visitors_reach_2015_2025.csv',
 'ffa_german_market_share_2001_2025.csv',
 'README.md',
 'ffa_visitors_age_gender_2021_2025.csv',
 'ffa_cinemas_arthouse_2016_2025.csv',
 'ffa_spending_cinema_homevideo_2003_2025.csv',
 'ffa_kinojahr_key_figures_2021_2025.csv',
 'ffa_cinema_svod_subscribers_2025.csv']

### 4: Load all four Q2 files

In [42]:
# Path to the raw data folder (read only, never modified!)
DATA = "../../data/raw/ffa"

# The four files of source Q2
files = [
    "ffa_spending_cinema_homevideo_2003_2025.csv",
    "ffa_visitors_age_gender_2021_2025.csv",
    "ffa_visitors_reach_2015_2025.csv",
    "ffa_cinema_svod_subscribers_2025.csv",
]

# Empty "Dictionary": we store each table under its file name
dfs = {}

#Read each file and print its size: (rows, columns)
#Tip: if a table has only 1 column, the seperator is wrong -> add sep=";" 
for f in files:
    dfs[f] = pd.read_csv(f"{DATA}/{f}") # dateiname und größe in Klammern 
    print(f, dfs[f].shape)

ffa_spending_cinema_homevideo_2003_2025.csv (23, 8)
ffa_visitors_age_gender_2021_2025.csv (5, 13)
ffa_visitors_reach_2015_2025.csv (11, 10)
ffa_cinema_svod_subscribers_2025.csv (2, 7)


##  5. File 1: Spending on cinema and home video (2003-2025)

### 5.1 First look at the table 

In [43]:
#Take one table from the dictionary and save under the short name "df" 
df = dfs["ffa_spending_cinema_homevideo_2003_2025.csv"]

#Show the first 5 rows: how is the table structured?
df.head()

,year,cinema_million_eur,dvd_bluray_vhs_million_eur,est_tvod_million_eur,svod_million_eur,total_million_eur,source_id,source
0,2003,846,1555,NaN,NaN,2401,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."
1,2004,886,1747,NaN,NaN,2632,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."
2,2005,738,1686,NaN,NaN,2424,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."
3,2006,807,1591,NaN,NaN,2398,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."
4,2007,755,1608,3.0,NaN,2367,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."


### 5.2: Columns, data types and missing values  

In [44]:
# Shows for each column: 
# - Non-Null Count: how many values exist (fewer than the number of rows= Lücken)
# -Dtype: int64 = whole number, float64 = decimal number, str/object = text
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   year                        23 non-null     int64  
 1   cinema_million_eur          23 non-null     int64  
 2   dvd_bluray_vhs_million_eur  23 non-null     int64  
 3   est_tvod_million_eur        19 non-null     float64
 4   svod_million_eur            11 non-null     float64
 5   total_million_eur           23 non-null     int64  
 6   source_id                   23 non-null     str    
 7   source                      23 non-null     str    
dtypes: float64(2), int64(4), str(2)
memory usage: 1.6 KB


### 5.3: Check: Does the total match the sum of the categories?

In [45]:
#Add up the single categories per row (axis=1 across columns, i.e. per year)
#Missing values (NaN) are treated as 0 here  
summe = df[["cinema_million_eur", "dvd_bluray_vhs_million_eur",
            "est_tvod_million_eur", "svod_million_eur"]].sum(axis=1)

#Helper column: how far does the reported total differ from our calculation?
#Small differences (-2 bis +2) = rounding, larger ones = errors 
df["difference"] = df["total_million_eur"] - summe

#Show only the relevant columns 
df[["year", "total_million_eur", "difference"]] 

,year,total_million_eur,difference
0,2003,2401,0.0
1,2004,2632,-1.0
2,2005,2424,0.0
3,2006,2398,0.0
4,2007,2367,1.0
5,2008,2363,0.0
6,2009,2633,0.0
7,2010,2573,0.0
8,2011,2652,0.0
9,2012,2712,0.0


### 5.4: From which year on are there streaming values?

In [46]:
#Seelct all rows where svod is NOT empty  (notna= "not NaN"), 
# and return the smalled year = first year with streaming data  

df.loc[df["svod_million_eur"].notna(), "year"].min()

np.int64(2015)

### 5.5: Remove the helper column

In [47]:
# The column "difference" was only needed fpr the check and is removed again
#Note: this only changes the table in the notebook, not the CSV file in data/raw/.
df = df.drop(columns="difference")

### 5.6 Findings: ffa_spending_cinema_homevideo_2003_2025.csv

 - **Size:** 23 rows (2003-2025, no years missing), 8 columns 
 - **Column names:** already English, lowercase, with underscored and the unit in the name (`_million_eur`) ✅ No action needed.
 - **Data types:** all numeric columns are int64 or float64, `year`is int64 ✅ 
    - `est_tvod`and `svod`are float64 only because they contain missing values.
 - **Missing values:**
    - `est_tvod_million_eur`: 4 missing (2003-2006), first value in 2007
    - `svod_million_eur`: 12 missing (2003-2014), first value in 2015, the first full year after the Netflix launch in Germany (September 2014)
    - These services did not exist yet or were not reported. A missing value (NaN) means "no value reported", not "0 euros".
    -**Cleaning decion:** keep as missing (NaN), do not replace with 0 (no silent imputation)-
 -  **Total check** `total_million_eur`matches the sum of all categories. Differences are only between -2 and +1 and are caused by rounding ✅ 
 -  **Intresting for subqestion 2:** In 2003, people spent almost twice as much on DVDs as on cinema (1,555 vs 846 million EUR).
 - **Metadeta columns:** `source_id` and `source` repeat the same value in every row.
    -**Decision:** drop both columns in the cleaned file.
    - They add no information for the analysis.
    - When we join all sources by `year`, every source would bring its own `source` column, which leads to duplicate column names.
    - The source is documented in the README and the data dictionary instead.


## 6. File 2: Visitors by age and gender (2021-2025) 

### 6.1 First look at the table 

In [48]:
#Tke the age and gender table from the dictionary 
df_age = dfs["ffa_visitors_age_gender_2021_2025.csv"]

#Show the first 5 rows (the file has only 5 rows, so this shows the whole table)
df_age.head()


,year,male_percent,female_percent,age_10_19_percent,age_20_29_percent,age_30_39_percent,age_40_49_percent,age_50_59_percent,age_60_plus_percent,tickets_base_million,avg_age,source_id,source
0,2021,53,47,19,14,20,17,16,13,40.1,38.3,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."
1,2022,50,50,21,14,18,17,15,15,74.0,38.8,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."
2,2023,49,51,21,15,16,17,15,16,91.3,38.9,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."
3,2024,49,51,22,14,16,17,14,17,85.1,39.2,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."
4,2025,48,52,20,13,16,17,16,19,88.6,40.6,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."


### 6.2 Columnns, data types and missing values 

In [49]:
# Data types and missing values of the age and gender table 
df_age.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   year                  5 non-null      int64  
 1   male_percent          5 non-null      int64  
 2   female_percent        5 non-null      int64  
 3   age_10_19_percent     5 non-null      int64  
 4   age_20_29_percent     5 non-null      int64  
 5   age_30_39_percent     5 non-null      int64  
 6   age_40_49_percent     5 non-null      int64  
 7   age_50_59_percent     5 non-null      int64  
 8   age_60_plus_percent   5 non-null      int64  
 9   tickets_base_million  5 non-null      float64
 10  avg_age               5 non-null      float64
 11  source_id             5 non-null      str    
 12  source                5 non-null      str    
dtypes: float64(2), int64(9), str(2)
memory usage: 652.0 bytes


### 6.3 Check: Do gender and age groups each ad up to 100%?

In [50]:
# Gender: male + female should be 100 in every year
df_age["gender_sum"] = df_age["male_percent"] + df_age["female_percent"]

# Age: the six age groups should add up to about 100 (rounding possible)
age_columns = ["age_10_19_percent", "age_20_29_percent", "age_30_39_percent",
               "age_40_49_percent", "age_50_59_percent", "age_60_plus_percent"]
df_age["age_sum"] = df_age[age_columns].sum(axis=1)

# Show the results
df_age[["year", "gender_sum", "age_sum"]]

,year,gender_sum,age_sum
0,2021,100,99
1,2022,100,100
2,2023,100,100
3,2024,100,100
4,2025,100,101


In [51]:
#remove the helper column again (only needed for the check)
df_age = df_age.drop(columns=["gender_sum", "age_sum"])

### 6.4 Findings: ffa_visitors_age_gender_2021_2025.csv

- **Size:** 5 rows (2021-2025, no years missing), 13 columns 
- **Column names:** already English, lowercase, with underscores and the unit in the name (`_percent`, `_million_`)
- **Data types / missing values:** (fill in from `info()`: eg. "all columns 5 non-null, no missing values; percentages int64; tickets_base_million and avg_age float64")
- **Structure:** two separate breakdowns in one table, not combined: 
    - Gender: `male_percent`, `female_percent`
    - Age: six groups (10-19, 20-29, 30-39, 40-49, 50-59, 60+). People under 10 are not included in the FFA panel.
    - Values are shares of **tickets** in percent, not shares of persons. 
- **Sum check:**
    - Gender: `male_percent` + `female_percent` = 100 in every year 
    - Age: the six age groups add up to 99-101 (2021: 99, 2025: 101). This is caused by rounding to whole percentages, not by an error.
    - **Decision:** keep the values as they are, do not adjust them. Differences of 1 percentage point between years should not be over-interpreted.
- **`tickets_base_million`:** ticket base of the FFA panel survey, not the official ticket count (2025: 88.6 vs. official 91.9 million). **Do not use as official ticket number**, use Q3 instead.
- **Interesting for sub-question 3:** the audience is getting older. `avg_age` rises from 38.3 (2021) to 40.6 (2025), and the share of 60+ rises from 13% to 19%.
- **Note:** 2021 is a COVID-19 year (cinemas closed until July), so the ticket base is very low (40.1 million).
- **Metadata columns:** `source_id` and `source`repeat the same value in every row -> drop in the cleaned file (same decision as file 1).

## 7. File 3: Cinema reach and visits per person (2015-2025)

### 7.1 First look at the table 

In [52]:
#Take the reach table from the dictionary 

df_reach = dfs["ffa_visitors_reach_2015_2025.csv"]#

#Show all rows (the file has only 11 rows)
df_reach.head(11) 

,year,cinema_visitors_million,cinema_reach_percent,visits_per_person,home_video_users_million,home_video_reach_percent,cinema_or_home_video_users_million,cinema_or_home_video_reach_percent,source_id,source
0,2015,29.9,44,4.5,23.8,35,39.2,58,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."
1,2016,25.0,37,4.7,23.9,35,36.4,54,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."
2,2017,25.0,37,4.7,22.7,34,35.7,53,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."
3,2018,24.6,37,4.1,23.0,34,35.5,53,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."
4,2019,28.2,38,4.0,26.1,35,41.1,55,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."
5,2020,16.8,23,2.2,28.1,38,36.8,49,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."
6,2021,16.8,23,2.4,28.1,38,36.4,49,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."
7,2022,21.7,29,3.4,28.8,39,39.4,53,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."
8,2023,23.8,32,3.8,29.6,40,41.2,56,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."
9,2024,21.6,29,3.9,30.6,41,40.6,54,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."


### 7.2  Columns, data types and missing Values 

In [53]:
# Data types and missing values of the reach table 
df_reach.info()

<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 10 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   year                                11 non-null     int64  
 1   cinema_visitors_million             11 non-null     float64
 2   cinema_reach_percent                11 non-null     int64  
 3   visits_per_person                   11 non-null     float64
 4   home_video_users_million            11 non-null     float64
 5   home_video_reach_percent            11 non-null     int64  
 6   cinema_or_home_video_users_million  11 non-null     float64
 7   cinema_or_home_video_reach_percent  11 non-null     int64  
 8   source_id                           11 non-null     str    
 9   source                              11 non-null     str    
dtypes: float64(4), int64(4), str(2)
memory usage: 1012.0 bytes


### 7.3 Check: Population base and plausibility 

In [54]:
# Check 1: Implied population base (in million)
# visitors/ reach * 100 = popilation in percentages refer to
# A jump from 2018 to 2019 shows the change in the survey population 

df_reach["implied_population_million"] = (df_reach["cinema_visitors_million"] / df_reach["cinema_reach_percent"] * 100)

# Check 2: visitors x visits per person = approximate number of tickets (in million)
# Can be compared with tickets_base_million from file 2 (2021: 40.1, 2025: 88.6)
df_reach["implied_tickets_million"] = (df_reach["cinema_visitors_million"] * df_reach["visits_per_person"])

df_reach[["year", "implied_population_million", "implied_tickets_million"]].round(1)

,year,implied_population_million,implied_tickets_million
0,2015,68.0,134.5
1,2016,67.6,117.5
2,2017,67.6,117.5
3,2018,66.5,100.9
4,2019,74.2,112.8
5,2020,73.0,37.0
6,2021,73.0,40.3
7,2022,74.8,73.8
8,2023,74.4,90.4
9,2024,74.5,84.2


In [55]:
# Remove the helper columns again 
df_reach = df_reach.drop(columns=["implied_population_million", "implied_tickets_million"])

### 7.4 Findings: ffa_visitors_reach_2015_2025.csv

- **Size:** 11 rows (2015-2025, no years missing), 10 columns 
- **Column names:** already English, lowercase, with underscores and the unit in the name (`_million`, `_percent`)
- **Data types / missing values:** all 10 columns have 11 non-null values, no missing values. `year` and all percentage columns are int64, the million columns and `visits_per_person` are float64 (decimal numbers).
- **Meaning of the columns:**
    -*Reach* = share of the population aged 10+ who used cinema or home video at least once in the year 
    -`cinema_or_home_video...` = people who used cinema, home video or both (counted once)
- **Method change 2018 -> 2019:** until 2018 only German citizens were surveyed, from 2019 all people living in Germany.
    - Visitors in million jump from 24.6 to 28.2, but reach only rises from 37% to 38%.
    - Check: the implied population base (visitors / reach x 100) is stable until 2018 and jumps in 2019. This confirms the method change.
    - **Decision:** for comparisions across 2019, use the **percentage columns**, not the million columns.
- **Suspicious values 2020/2021:** four values are identical in both years ( `cinema_visitors_million` 16.8, `cinema_reach_percent` 23, `home_video_users_million` 28.1, `home_video_reach_percent` 38), while `visits_per_person` and `cinema_or_home_video_users_million` differ.
    -Check: visitors x visits per person gives 40.3 million tickets in 2021, which matches the ticket base in file 2 (40.1). So the 2021 row is consistent. 2020 cannot be checked this way.
    -PDF check: the identical values fpr 2020 and 2021 are confirmed in the FFA PDF (cinema visitors and reach: p.16, home video users and reach: p.58). The FFA reports the same values for both years, so this is not a copy error. No correction needed. 
- **Plausibility check:** visitors x visits per person matches the ticket base in file 2 for 2021 (40.3 vs. 40.1) and 2025 (88.9 vs. 88.6)
- **Metadata columns:** `source_id` and `source` repeat the same value in every row -> drop in the cleaned file (same decision as files 1 and 2)


## 8. File 4: Cinema visitors and straming subscribers (2025)

### 8.1 First look at the table 

In [56]:
# Take the streaming subscribers table from the dictionary 

df_svod= dfs["ffa_cinema_svod_subscribers_2025.csv"]

# Show the whole table (it has only 2 rows)
df_svod

,year,group,cinema_reach_percent,tickets_per_person,avg_ticket_price_eur,source_id,source
0,2025,population_10_plus,31,3.8,10.39,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."
1,2025,svod_subscribers,36,4.8,10.48,Q2,"Filmförderungsanstalt (FFA), Kinobesucher*inne..."


### 8.2 Columns, data types and missing values

In [57]:
# Data types and missing values of the svod table 
df_svod.info()

<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   year                  2 non-null      int64  
 1   group                 2 non-null      str    
 2   cinema_reach_percent  2 non-null      int64  
 3   tickets_per_person    2 non-null      float64
 4   avg_ticket_price_eur  2 non-null      float64
 5   source_id             2 non-null      str    
 6   source                2 non-null      str    
dtypes: float64(2), int64(2), str(3)
memory usage: 244.0 bytes


### 8.3 Findings: ffa_cinema_svod_subscribers_2025.csv

- **Size:** 2 rows, 7 columns. Only one year (2025), two groups: `population_10_plus` and `svod_subscribers`.
- **Column names:** already English, lowercase, with underscores and the unit in the name (`_percent`, `_eur`)
- **Data types / missing values:** all 7 columns have 2 non-null values, no missing values. `year` and `cinema_reach_percent` are int64, `tickets_per_person` and `avg_ticket_price_eur` are float64, `group` is text (str)
- **Structure:** different from files 1-3. Rows are **groups**, not years. So this file cannot be joined with the other sources by `year` in the same way. 
- **Content:** SVOD subscribers go to the cinema more often the average population (reach 36% vs. 31%, 4.8 vs. 3.8 tickets per person)
    - Relevant for sub-question 2: suggests that streaming and cinema coexist rather than streaming replacing cinema.
    - Limitations: only a snapshot for 2025, no trend and no statistical test possible. Correlation is not causation (subscribers may differ in age or interest in films).
- **Plausibility check:** the `population_10_plus` row (31%, 3.8 tickets per person) matches the 2025 values in file 3
- **Decision:** not part of the joined dataset by year. Use as a single supporting fact for sub-question 2 in the analysis and presentation.
- **Metadata columns:** `source_id` and `source` repeat the same value in every row -> drop (same decision as files 1-3)